# PART 10. 비즈니스 전략 — 분석과 모델을 실행으로

> **핵심 질문: 지금까지 만든 분석과 모델로, 내일 무엇을 할 수 있을까?**

## 📋 한눈에 보기

**작업 흐름**
```
1. PART 5~9 결과 한 장 요약
2. 고객 기반 모델을 안 한 이유 확인
3. 저장한 모델 점검 (STEP2 준비)
4. 최신 시점(2018-08-23)에 모델 적용
5. 이탈 위험 × 판매액으로 우선순위
6. 별점과 내용이 다른 리뷰 찾기
7. 전략 제안 · 한계 정리
```

| 항목 | 내용 |
|---|---|
| 사용 데이터 | 원본 CSV + PART 6 · 7-1 · 8 · 9에서 저장한 모델 |
| 결과물 | 실행 목록 (연락할 판매자, 확인할 리뷰) + 전략 표 |

**🎯 핵심 (꼭 가져갈 것)**
- **데이터가 문제를 정합니다.** 재구매 3% → 고객 대신 상품(추천)과 판매자(이탈)로 문제를 다시 정의했습니다.
- 우선순위는 **확률 × 영향**(이탈 확률 × 판매액)으로 정합니다.

**💡 인사이트 (이 파트에서 확인한 사실)**
- 고객 × 상품 표에서 채워진 칸은 0.003% → 협업 필터링을 할 수 없습니다.
- 배포용 모델 4개 합계 약 330KB → 작은 서버에서도 서빙할 수 있습니다.
- 별점 4~5점인데 내용은 강한 불만인 리뷰 23건을 찾았습니다(예: "아직 상품을 못 받았다").

---

## 수업 목표
- PART 5~9의 결과를 한 장으로 정리합니다.
- 이 데이터로 **고객 기반 추천·고객 이탈 예측을 하지 않은 이유**를 숫자로 확인합니다.
- 저장한 모델 파일을 점검합니다. (STEP2 서빙 준비)
- 가장 최근 데이터에 모델을 적용해 **실행할 대상 목록**을 만들고, 전략으로 연결합니다.

## 수업 진행 포인트
| 구분 | 설명 |
|---|---|
| 데이터 관점 | 모델을 학습한 시점(2018-06-01)이 아니라 **데이터의 마지막 시점**에 모델을 적용합니다. |
| 코드 관점 | 저장한 모델을 `joblib.load()`로 불러와 새 데이터에 적용합니다. 학습 코드는 없습니다. |
| AI 관점 | 모델의 가치는 점수가 아니라 **어떤 행동을 바꾸는가**에 있습니다. |

---
## 1. 지금까지의 여정

| PART | 질문 | 방법 | 핵심 결과 | 비즈니스 활용 |
|---|---|---|---|---|
| 5 EDA | 데이터에서 무엇이 보이나? | 집계·시각화 | 재구매 3%, 배송 지연 → 리뷰 급락 | 문제 정의 |
| 6 유사 상품 추천 | 같은 카테고리면 비슷할까? | 규칙 vs KNN | KNN이 가격·무게·부피를 고르게 맞춤 | 상품 상세 페이지 "비슷한 상품" |
| 7-1~7-3 리뷰 감성 | 글만 보고 불만을 알 수 있나? | TF-IDF · 딥러닝 · BERT | 세 모델 모두 정확도 약 0.92 안팎, 학습 없이 쓴 BERT는 더 낮음 | 부정 리뷰 자동 분류 |
| 8 판매자 유형 | 어떤 판매자들이 있나? | K-Means | 핵심 우량 · 신규 성장 · 휴면 · 저평점 위험 | 유형별 관리 정책 |
| 9 판매자 이탈 | 누가 판매를 멈출까? | 규칙 vs 로지스틱 · 랜덤 포레스트 | 로지스틱이 규칙보다 꾸준히 조금 나음, 이탈 확률 제공 | 이탈 위험 판매자 선제 관리 |

> 📌 공통 메시지 : **단순한 방법과 먼저 비교하고, 나아질 때 복잡한 방법을 쓴다.**

In [14]:
# 셀 1. Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# 셀 2. 라이브러리 불러오기

# 그래프 한글 깨짐 방지 (Colab에는 한글 폰트가 없어서 □□□로 표시됨)
!pip install -q koreanize-matplotlib

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib  # import만 하면 한글 폰트가 자동 적용됩니다

base_path  = "/content/drive/MyDrive/Colab Notebooks/Data_Analysis/Olist"
data_path  = f"{base_path}/data"     # PART 2에서 저장한 원본 CSV 위치
model_path = f"{base_path}/model"    # 학습한 모델을 저장할 위치
os.makedirs(model_path, exist_ok=True)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("경로 설정 완료!")

경로 설정 완료!


---
## 2. 왜 고객 기반 추천 · 고객 이탈 예측을 하지 않았을까?

PART 5에서 "이 데이터로 고객 기반 추천을 하지 않은 이유는 PART 10에서 확인합니다"라고 했습니다. 숫자로 확인해 봅니다.

In [16]:
# 셀 3. 원본 데이터 불러오기

date_cols = ["order_purchase_timestamp", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
orders  = pd.read_csv(f"{data_path}/olist_orders_dataset.csv", parse_dates=date_cols)
items   = pd.read_csv(f"{data_path}/olist_order_items_dataset.csv", parse_dates=["shipping_limit_date"])
reviews = pd.read_csv(f"{data_path}/olist_order_reviews_dataset.csv", parse_dates=["review_creation_date"])
sellers = pd.read_csv(f"{data_path}/olist_sellers_dataset.csv")

print(f"주문 {len(orders):,}건 / 판매 상품 {len(items):,}행 / 리뷰 {len(reviews):,}건 / 판매자 {len(sellers):,}명")
customers = pd.read_csv(f"{data_path}/olist_customers_dataset.csv")

주문 99,441건 / 판매 상품 112,650행 / 리뷰 99,224건 / 판매자 3,095명


In [17]:
# 셀 4. 고객 데이터로 추천·이탈 예측이 가능할까?

valid = orders[~orders["order_status"].isin(["canceled", "unavailable"])]     # 유효 주문
cust = valid.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
n_orders = cust.groupby("customer_unique_id")["order_id"].nunique()             # 고객(실제 사람)별 주문 수

# 고객 × 상품 표 : 칸 하나 = "이 고객이 이 상품을 샀다". 협업 필터링은 이 표의 빈칸을 채우는 방법입니다.
pairs = items.merge(cust[["order_id", "customer_unique_id"]], on="order_id")[["customer_unique_id", "product_id"]].drop_duplicates()
n_c, n_p = pairs["customer_unique_id"].nunique(), pairs["product_id"].nunique()
density = len(pairs) / (n_c * n_p)      # 채워진 칸 비율

print(f"고객 수                : {len(n_orders):,}명")
print(f"한 번만 구매한 고객    : {(n_orders == 1).mean():.1%}")
print(f"고객 1명이 산 상품 종류 (평균) : {pairs.groupby('customer_unique_id').size().mean():.2f}개")
print(f"고객 × 상품 표에서 채워진 칸   : {density:.5%}  ({n_c:,}명 × {n_p:,}개)")

고객 수                : 94,990명
한 번만 구매한 고객    : 97.0%
고객 1명이 산 상품 종류 (평균) : 1.07개
고객 × 상품 표에서 채워진 칸   : 0.00327%  (94,983명 × 32,729개)


**해석**

| 하려던 것 | 필요한 데이터 | Olist의 현실 | 그래서 |
|---|---|---|---|
| 고객 기반 추천 (협업 필터링) | "비슷한 고객이 산 상품" → 고객마다 여러 상품 구매 이력 | 고객 1명이 평균 1개 남짓 구매, 고객×상품 표가 거의 빈칸 | **상품 특징 기반 추천 (PART 6)** |
| 고객 이탈 예측 | "돌아온 고객 vs 안 돌아온 고객" 예시 | 97%가 한 번 사고 끝 → 거의 전부 '이탈' | **판매자 이탈 예측 (PART 9)** |

> 💡 데이터가 문제를 정합니다. **"무엇을 하고 싶은가"보다 "데이터가 무엇을 허락하는가"를 먼저 확인**하고, 문제를 다시 정의하는 것이 실무의 첫 단계입니다.

---
## 3. 저장한 모델 점검 (STEP2 서빙 준비)

| 파일 | 만든 곳 | 용도 |
|---|---|---|
| part6_recommend.pkl | PART 6 | 유사 상품 추천 |
| part7_1_sentiment_ml.pkl | PART 7-1 | 리뷰 감성 분류 **(배포용)** |
| part7_2_sentiment_model.keras · part7_2_tokenizer.pkl | PART 7-2 | 리뷰 감성 분류 (딥러닝, 비교용) |
| part8_seller_segment.pkl | PART 8 | 판매자 유형 판단 |
| part9_seller_churn.pkl | PART 9 | 판매자 이탈 확률 |

> BERT(PART 7-3)는 저장하지 않습니다. 다운로드만 약 540MB이고, 이 데이터에서는 PART 7-1 모델이 더 가볍고 정확했습니다.

In [18]:
# 셀 5. 모델 파일 점검 : 있는가? 크기는?

model_files = [
    ("part6_recommend.pkl",           "PART 6",   "유사 상품 추천",        "배포"),
    ("part7_1_sentiment_ml.pkl",      "PART 7-1", "리뷰 감성 (머신러닝)",  "배포"),
    ("part7_2_sentiment_model.keras", "PART 7-2", "리뷰 감성 (딥러닝)",    "비교용"),
    ("part7_2_tokenizer.pkl",         "PART 7-2", "딥러닝 단어집",         "비교용"),
    ("part8_seller_segment.pkl",      "PART 8",   "판매자 유형",           "배포"),
    ("part9_seller_churn.pkl",        "PART 9",   "판매자 이탈 확률",      "배포"),
]
rows = []
for name, part, use, deploy in model_files:
    path = f"{model_path}/{name}"
    ok = os.path.exists(path)
    rows.append({"파일": name, "PART": part, "용도": use, "구분": deploy,
                 "상태": "✅ 있음" if ok else "❌ 없음 (해당 PART 실행 필요)",
                 "크기(KB)": round(os.path.getsize(path) / 1024) if ok else None})
inventory = pd.DataFrame(rows)
total = inventory.loc[inventory["구분"] == "배포", "크기(KB)"].sum()
print(f"배포용 모델 4개 합계 : {total:,.0f} KB")
inventory

배포용 모델 4개 합계 : 340 KB


,파일,PART,용도,구분,상태,크기(KB)
0,part6_recommend.pkl,PART 6,유사 상품 추천,배포,✅ 있음,159
1,part7_1_sentiment_ml.pkl,PART 7-1,리뷰 감성 (머신러닝),배포,✅ 있음,176
2,part7_2_sentiment_model.keras,PART 7-2,리뷰 감성 (딥러닝),비교용,✅ 있음,2541
3,part7_2_tokenizer.pkl,PART 7-2,딥러닝 단어집,비교용,✅ 있음,463
4,part8_seller_segment.pkl,PART 8,판매자 유형,배포,✅ 있음,3
5,part9_seller_churn.pkl,PART 9,판매자 이탈 확률,배포,✅ 있음,2


**해석**: 배포용 모델 4개를 모두 합쳐도 수백 KB 수준입니다. 작은 서버나 무료 클라우드에서도 바로 서빙할 수 있습니다.

---
## 4. 가장 최근 데이터에 모델 적용하기

PART 8·9 모델은 **2018-06-01 기준**으로 학습했습니다. 실제로 쓸 때는 **가장 최근 시점**의 판매자 정보를 만들어 모델에 넣습니다.

| 항목 | 값 |
|---|---|
| 적용 기준일 | **2018-08-23** (8월 22일까지의 정보) |
| 이유 | 원본 데이터는 8월 23일 무렵부터 주문 수가 급격히 줄어듭니다(수집 종료 구간). 이 구간을 넣으면 멀쩡한 판매자도 "최근 판매 없음"으로 보입니다. |
| 예측 대상 | 6~8월(기준일 직전 3개월)에 판매가 있었던 판매자 |
| 예측 내용 | 이후 3개월(8월 말 ~ 11월) 동안 주문이 없을 확률 |

> ⚠️ 이 예측이 맞았는지는 확인할 수 없습니다. 데이터가 여기서 끝나기 때문입니다. **실제 서비스에서 모델을 쓰는 상황이 바로 이것**입니다. 정답을 모르는 미래를 예측하고, 3개월 뒤 실제 결과와 비교해 모델을 점검합니다.

In [19]:
# 셀 6. 판매자 프로필 함수 (판매자 1명 = 1행)
# ⚠️ 기준일(cutoff) '이전'에 알 수 있었던 정보만 사용합니다.
#    - 기준일 전에 들어온 주문만
#    - 배송 완료·출고 시각도 기준일 전에 일어난 것만
#    - 리뷰도 기준일 전에 작성된 것만
# PART 8 · PART 9 · PART 10(배포)에서 똑같은 함수를 씁니다.

def make_seller_profile(orders, items, reviews, cutoff, start="2017-01-01"):
    cutoff = pd.Timestamp(cutoff)
    o = orders[
        (orders["order_purchase_timestamp"] >= start)
        & (orders["order_purchase_timestamp"] < cutoff)
        & ~orders["order_status"].isin(["canceled", "unavailable"])   # 유효 주문만
    ]
    line = items.merge(o, on="order_id")

    # 판매자-주문 단위로 정리 (한 주문에 같은 판매자 상품이 여러 개면 1행으로)
    so = line.groupby(["seller_id", "order_id"]).agg(
        purchase=("order_purchase_timestamp", "first"),
        carrier=("order_delivered_carrier_date", "first"),     # 택배사에 넘긴 시각 (출고)
        delivered=("order_delivered_customer_date", "first"),  # 고객이 받은 시각
        estimated=("order_estimated_delivery_date", "first"),  # 안내한 도착 예정일
        limit=("shipping_limit_date", "max"),                  # 판매자 출고 마감 시각
        sales=("price", "sum"),
    ).reset_index()

    # 기준일 이후에 일어난 일은 '아직 모르는 정보' → 빈 값으로
    so["carrier"]   = so["carrier"].where(so["carrier"] < cutoff)
    so["delivered"] = so["delivered"].where(so["delivered"] < cutoff)
    score = reviews[reviews["review_creation_date"] < cutoff].groupby("order_id")["review_score"].mean()
    so["score"] = so["order_id"].map(score)

    # 주문별 지표 만들기 (모르는 값은 빈 값(NaN)으로 두고, 평균 낼 때 자동으로 빠집니다)
    so["delivery_days"]  = (so["delivered"] - so["purchase"]).dt.total_seconds() / 86400   # 주문 → 수령까지 걸린 일수
    so["is_late"]        = np.where(so["delivered"].isna(), np.nan, so["delivered"] > so["estimated"])  # 예정일보다 늦게 도착? (1/0)
    so["is_ship_late"]   = np.where(so["carrier"].isna(), np.nan, so["carrier"] > so["limit"])          # 출고 마감보다 늦게 출고? (1/0)
    so["is_low_review"]  = np.where(so["score"].isna(), np.nan, so["score"] <= 2)                        # 1~2점 리뷰? (1/0)
    so["month"]          = so["purchase"].dt.to_period("M")                                             # 주문한 연-월 (예: 2018-03)

    # 판매자 1명 = 1행으로 묶기

    g = so.groupby("seller_id")
    prof = pd.DataFrame({
        "frequency":         g.size(),                                         # F : 주문 건수
        "monetary":          g["sales"].sum(),                                 # M : 판매액 (R$)
        "recency_days":      (cutoff - g["purchase"].max()).dt.days,           # R : 마지막 판매 후 일수
        "tenure_days":       (cutoff - g["purchase"].min()).dt.days,           # 첫 판매 후 일수
        "active_months":     g["month"].nunique(),                             # 판매가 있었던 달 수
        "avg_delivery_days": g["delivery_days"].mean(),                        # 평균 배송 일수
        "late_rate":         g["is_late"].mean(),                              # 고객 도착 지연 비율
        "ship_late_rate":    g["is_ship_late"].mean(),                         # 판매자 출고 지연 비율
        "avg_review_score":  g["score"].mean(),                                # 평균 리뷰 점수
        "low_review_rate":   g["is_low_review"].mean(),                        # 1~2점 리뷰 비율
    })
    prof["avg_order_value"] = prof["monetary"] / prof["frequency"]             # 주문당 평균 금액
    return prof.reset_index()

In [20]:
# 셀 7. 이탈 예측용 특징 함수 + 정답(라벨) 함수
# 특징(X)은 기준일 '이전' 정보만, 정답(y)은 기준일 '이후' 3개월 정보로 만듭니다.

def count_orders(orders, items, start, end):
    """start 이상 ~ end 미만 기간에 판매자별로 유효 주문이 몇 건인지 셉니다."""
    valid = orders[~orders["order_status"].isin(["canceled", "unavailable"])]
    so = items[["order_id", "seller_id"]].drop_duplicates().merge(
        valid[["order_id", "order_purchase_timestamp"]], on="order_id")
    t = so["order_purchase_timestamp"]
    return so[(t >= start) & (t < end)].groupby("seller_id").size()

def make_churn_features(orders, items, reviews, cutoff):
    """기준일 직전 3개월에 판매가 있었던 판매자의 특징 (판매자 1명 = 1행)"""
    cutoff = pd.Timestamp(cutoff)
    m3 = cutoff - pd.DateOffset(months=3)   # 기준일 3개월 전
    m6 = cutoff - pd.DateOffset(months=6)   # 기준일 6개월 전
    prof = make_seller_profile(orders, items, reviews, cutoff)
    prof["orders_last_3m"] = prof["seller_id"].map(count_orders(orders, items, m3, cutoff)).fillna(0)
    prof["orders_prev_3m"] = prof["seller_id"].map(count_orders(orders, items, m6, m3)).fillna(0)
    prof["order_trend"]    = prof["orders_last_3m"] - prof["orders_prev_3m"]   # +면 늘어나는 중

    # 모델 입력용 : 한쪽으로 크게 치우친 값(주문 수·금액)은 로그로 폅니다.
    # (로지스틱 회귀는 직선 관계를 가정하므로, 아주 큰 판매자 몇 명에게 예측이 끌려가지 않도록)
    for col in ["frequency", "monetary", "avg_order_value", "orders_last_3m", "orders_prev_3m"]:
        prof[f"log_{col}"] = np.log1p(prof[col])
    prof["log_order_trend"] = prof["log_orders_last_3m"] - prof["log_orders_prev_3m"]   # 증감 비율
    return prof[prof["orders_last_3m"] > 0].reset_index(drop=True)            # 최근 활동한 판매자만

def add_churn_label(df, orders, items, cutoff, months=3):
    """기준일 이후 months개월 동안 유효 주문이 0건이면 이탈(1)"""
    cutoff = pd.Timestamp(cutoff)
    future = count_orders(orders, items, cutoff, cutoff + pd.DateOffset(months=months))
    df = df.copy()
    df["churn"] = (df["seller_id"].map(future).fillna(0) == 0).astype(int)
    return df

# 군집에 쓰는 특징 (PART 8에서 정한 것)
seg_features = ["log_frequency", "log_monetary", "recency_days", "tenure_days", "avg_review_score"]

def make_segment_input(prof, review_fill):
    """판매자 프로필 → 군집 입력 (로그 변환 + 결측 채우기)"""
    return pd.DataFrame({
        "log_frequency":    np.log1p(prof["frequency"]),
        "log_monetary":     np.log1p(prof["monetary"]),
        "recency_days":     prof["recency_days"],
        "tenure_days":      prof["tenure_days"],
        "avg_review_score": prof["avg_review_score"].fillna(review_fill),
    })

In [21]:
# 셀 8. 최신 기준일로 판매자 특징 만들기 + 유형 · 이탈 확률 예측

NOW = "2018-08-23"
latest = make_churn_features(orders, items, reviews, NOW)
print(f"최근 3개월(6~8월) 활동 판매자 : {len(latest):,}명")

seg_file, churn_file = f"{model_path}/part8_seller_segment.pkl", f"{model_path}/part9_seller_churn.pkl"
if os.path.exists(seg_file) and os.path.exists(churn_file):
    seg, churn = joblib.load(seg_file), joblib.load(churn_file)

    # PART 8 모델 : 판매자 유형
    Xs = make_segment_input(latest, seg["review_fill"])[seg["features"]]
    latest["segment"] = pd.Series(seg["kmeans"].predict(seg["scaler"].transform(Xs))).map(seg["names"])

    # PART 9 모델 : 이탈 확률
    latest["churn_prob"] = churn["model"].predict_proba(latest[churn["features"]])[:, 1]
    latest["risk"] = pd.cut(latest["churn_prob"], [0, 0.3, 0.5, 0.7, 1.0],
                            labels=["낮음", "보통", "높음", "매우 높음"], include_lowest=True)
    print(f"이탈 위험(확률 0.5 이상) 판매자 : {(latest['churn_prob'] >= churn['threshold']).sum():,}명")
    display(latest["risk"].value_counts().sort_index().to_frame("판매자 수"))
else:
    print("PART 8, PART 9를 먼저 실행해 모델 파일을 만들어 주세요.")

최근 3개월(6~8월) 활동 판매자 : 1,817명
이탈 위험(확률 0.5 이상) 판매자 : 637명


,판매자 수
risk,
낮음,742
보통,438
높음,332
매우 높음,305


**해석**: 확률 0.5 이상인 판매자가 PART 9 학습 데이터의 실제 이탈률(약 24%)보다 많이 잡힙니다. PART 9에서 `class_weight="balanced"`로 **이탈을 놓치지 않는 쪽**으로 학습했기 때문입니다(재현율 ↑, 정밀도 ↓). 담당자가 연락할 수 있는 인원이 정해져 있다면 기준(0.5)을 올리거나, 아래처럼 **우선순위로 줄을 세워** 위에서부터 처리합니다.

In [22]:
# 셀 9. 유형 × 이탈 위험 : 누구부터 챙겨야 할까?

if "churn_prob" in latest.columns:
    matrix = pd.crosstab(latest["segment"], latest["risk"])
    order = [s for s in ["핵심 우량", "신규 성장", "휴면", "저평점 위험"] if s in matrix.index]
    display(matrix.loc[order])

    # 최근 3개월(6~8월) 실제 판매액
    v = orders[~orders["order_status"].isin(["canceled", "unavailable"])]
    sales = items.merge(v[["order_id", "order_purchase_timestamp"]], on="order_id")
    t = sales["order_purchase_timestamp"]
    sales = sales[(t >= pd.Timestamp(NOW) - pd.DateOffset(months=3)) & (t < NOW)]
    latest["sales_last_3m"] = latest["seller_id"].map(sales.groupby("seller_id")["price"].sum()).fillna(0)

    # 예상 손실 = 이탈 확률 × 최근 3개월 판매액  → 확률과 영향을 함께 봅니다
    latest["expected_loss"] = latest["churn_prob"] * latest["sales_last_3m"]
    top = latest.sort_values("expected_loss", ascending=False).head(10)
    display(pd.DataFrame({
        "seller_id": top["seller_id"].str[:8],
        "유형": top["segment"],
        "이탈확률": top["churn_prob"].round(2),
        "최근3개월주문": top["orders_last_3m"].astype(int),
        "마지막판매후일수": top["recency_days"],
        "최근3개월판매액(R$)": top["sales_last_3m"].round(0),
        "예상손실(R$)": top["expected_loss"].round(0),
    }).reset_index(drop=True))

risk,낮음,보통,높음,매우 높음
segment,,,,
핵심 우량,498,149,106,104
신규 성장,244,288,219,173
휴면,0,0,1,4
저평점 위험,0,1,6,24


,seller_id,유형,이탈확률,최근3개월주문,마지막판매후일수,최근3개월판매액(R$),예상손실(R$)
0,b37c4c02,핵심 우량,0.92,1,38,"7,160.00","6,618.00"
1,40db9e9a,핵심 우량,0.33,9,5,"17,728.00","5,849.00"
2,80ceebb4,신규 성장,0.86,1,28,"6,729.00","5,773.00"
3,8fb79102,신규 성장,0.46,5,31,"8,120.00","3,734.00"
4,2bf6a2c1,핵심 우량,0.25,8,9,"13,894.00","3,502.00"
5,52f976b1,핵심 우량,0.58,4,30,"5,030.00","2,941.00"
6,2eb70248,핵심 우량,0.17,80,34,"17,167.00","2,916.00"
7,b839e417,핵심 우량,0.11,22,2,"27,118.00","2,865.00"
8,c4d51195,핵심 우량,0.76,6,78,"3,704.00","2,822.00"
9,87e24fc7,신규 성장,0.41,6,8,"6,824.00","2,795.00"


In [23]:
cols = ["seller_id", "frequency", "monetary", "orders_last_3m", "orders_prev_3m", "sales_last_3m"] + churn["features"]
latest[cols].to_csv(f"{model_path}/seller_table.csv", index=False)

**해석**
- 위 표는 **예상 손실 = 이탈 확률 × 최근 3개월 판매액**이 큰 순서입니다.
- 이탈 확률만으로 줄을 세우면 판매액이 작은 판매자에게 먼저 시간을 쓰게 됩니다. 판매액만으로 세우면 떠날 생각이 없는 판매자에게 시간을 씁니다. **확률과 영향을 함께** 보는 것이 실무 방식입니다.
- '핵심 우량' 유형인데 위험이 높다면 최우선 관리 대상입니다.

In [24]:
# 셀 10. 최근 리뷰에 감성 모델 적용 : 별점과 내용이 다른 리뷰 찾기

ml_file = f"{model_path}/part7_1_sentiment_ml.pkl"
if os.path.exists(ml_file):
    ml = joblib.load(ml_file)
    recent = reviews[(reviews["review_creation_date"] >= "2018-08-01")
                     & reviews["review_comment_message"].notna()].drop_duplicates("review_id").copy()
    recent["neg_prob"] = ml["model"].predict_proba(ml["tfidf"].transform(recent["review_comment_message"]))[:, 0]
    recent["pred"] = np.where(recent["neg_prob"] >= 0.5, "부정", "긍정")

    print(f"2018-08 이후 글이 있는 리뷰 : {len(recent):,}건")
    print(f"  별점 1~2점 비율        : {(recent['review_score'] <= 2).mean():.1%}")
    print(f"  모델이 '부정'으로 본 비율 : {(recent['pred'] == '부정').mean():.1%}")

    # 별점은 4~5점인데 글은 강하게 부정 → 별점만 보는 대시보드에서는 놓치는 불만
    mismatch = recent[(recent["review_score"] >= 4) & (recent["neg_prob"] >= 0.9)]
    print(f"\n별점 4~5점인데 모델이 강한 부정(확률 0.9 이상)으로 본 리뷰 : {len(mismatch):,}건")
    pd.set_option("display.max_colwidth", 90)
    display(mismatch.sort_values("neg_prob", ascending=False)[["review_score", "neg_prob", "review_comment_message"]].head(5))
else:
    print("PART 7-1을 먼저 실행해 part7_1_sentiment_ml.pkl을 만들어 주세요.")

2018-08 이후 글이 있는 리뷰 : 3,471건
  별점 1~2점 비율        : 23.3%
  모델이 '부정'으로 본 비율 : 33.1%

별점 4~5점인데 모델이 강한 부정(확률 0.9 이상)으로 본 리뷰 : 23건


,review_score,neg_prob,review_comment_message
74668,4,1.00,"Faltou a nota fiscal que não chegou com o produto ,e nao esta no email, que informa \r..."
72891,4,0.99,AINDA NÃO RECEBI O PRODUTO
78354,4,0.99,Comprei preto veio azul porém não está funcionado foi 1 que veio com defeito
2742,4,0.98,não obtive retorno até o momento
75787,4,0.98,"Este produto veio certo, mas comprei uma coifa e quero devolver e não estou conseguind..."


**해석**
- 모델이 '부정'으로 본 비율이 실제 1~2점 비율보다 높습니다. PART 7-1에서 `class_weight="balanced"`로 부정을 놓치지 않게 학습했고, 학습 때 뺐던 3점 리뷰도 여기서는 함께 분류했기 때문입니다.
- 별점은 높은데 내용은 불만인 리뷰(예: "아직 상품을 못 받았다", "다른 색이 왔고 불량이다")는 **별점만 보는 대시보드에서는 보이지 않습니다.** 감성 모델이 이런 리뷰를 CS팀에 넘겨 줄 수 있습니다.
- 반대로 모델이 틀린 경우도 섞여 있습니다(예: "não tive problema" = 문제없었다). 자동 분류는 **사람이 확인할 목록을 줄여 주는 도구**로 씁니다.

In [25]:
# 셀 11. 유사 상품 추천 모델 적용 : 가장 많이 팔린 상품에 붙일 "비슷한 상품"

rec_file = f"{model_path}/part6_recommend.pkl"
if os.path.exists(rec_file):
    art = joblib.load(rec_file)
    prods = art["products"]

    def recommend(product_id):
        hit = prods.index[prods["product_id"] == product_id]
        if len(hit) == 0:
            return []
        i = hit[0]
        x = art["scaler"].transform(np.log1p(prods.loc[[i], art["features"]]))
        nn, cat_idx = art["knn_by_category"][prods.at[i, "category"]]
        _, pos = nn.kneighbors(x, n_neighbors=art["k"] + 1)
        rec = cat_idx[pos[0]]
        return list(rec[rec != i][:art["k"]])

    sold = items[items["product_id"].isin(prods["product_id"])]["product_id"].value_counts()
    base_id = sold.index[0]
    i = prods.index[prods["product_id"] == base_id][0]
    rec_idx = recommend(base_id)
    t = prods.loc[[i] + rec_idx, ["product_id", "category_kr", "price", "weight_g", "photos_qty", "avg_review_score"]].copy()
    t.insert(0, "구분", ["기준 상품 (판매 1위)"] + [f"추천 {k+1}" for k in range(len(rec_idx))])
    t["product_id"] = t["product_id"].str[:8]
    display(t.reset_index(drop=True))
else:
    print("PART 6을 먼저 실행해 part6_recommend.pkl을 만들어 주세요.")

,구분,product_id,category_kr,price,weight_g,photos_qty,avg_review_score
0,기준 상품 (판매 1위),2b4609f8,건강·뷰티,87.36,250.00,3.00,4.07
1,추천 1,20969961,건강·뷰티,72.97,200.00,2.00,4.33
2,추천 2,189e539d,건강·뷰티,143.40,300.00,2.00,3.75
3,추천 3,681953cc,건강·뷰티,74.62,150.00,3.00,4.00
4,추천 4,4c7d4a2e,건강·뷰티,36.19,250.00,3.00,4.38
5,추천 5,076e4ce5,건강·뷰티,104.99,633.00,2.00,4.33


---
## 5. 전략 제안 : 문제 → 근거 → 실행 → 측정

| 문제 (PART 5) | 근거 (모델) | 실행 | 측정 지표 (KPI) |
|---|---|---|---|
| 재구매 3%, 성장이 신규 유입에만 의존 (인사이트 1·5) | PART 6 유사 상품 추천 | 상품 페이지·장바구니에 "비슷한 상품" 노출 → 객단가·추가 구매 유도 | 추천 클릭률, 주문당 상품 수, 객단가 |
| 배송 지연 → 리뷰 급락 (인사이트 9) | PART 7-1 감성 분류 | 부정 리뷰 자동 분류 → CS팀 우선 대응, 별점 높은 불만 리뷰도 포착 | 부정 리뷰 응답 시간, 재리뷰 평점 |
| 지연·저평점이 일부 판매자에게 몰림 (인사이트 9·10) | PART 8 판매자 유형 | 저평점 위험 → 출고 마감 준수 관리 / 신규 성장 → 온보딩 / 핵심 우량 → 우수 판매자 혜택 | 유형별 출고 지연율, 평균 평점 |
| 판매자가 떠나면 상품 구색과 매출이 줄어듦 | PART 9 이탈 확률 | 확률 × 판매액으로 우선순위 → 담당자 연락, 광고·수수료 지원 | 3개월 뒤 실제 이탈률 (모델 점검), 유지된 판매액 |

## 6. 한계와 다음 단계

| 한계 | 다음 단계 |
|---|---|
| 추천은 특징 유사도만 쟀을 뿐, 고객 반응은 모름 | 서비스에 붙인 뒤 A/B 테스트로 클릭률·구매 전환율 확인 |
| 이탈 정의는 실습용 (3개월 주문 없음) | 실제 판매자 탈퇴·휴면 정책과 맞춰 정의 다시 세우기 |
| 모델은 2018-06 시점으로 학습 | 3개월마다 새 데이터로 다시 학습하고 성능 비교 |
| 감성 모델은 포르투갈어 리뷰로만 학습 | 다른 언어·새 표현이 늘면 데이터 추가 후 재학습 |

> 👉 **STEP2(서빙)**에서는 여기서 점검한 모델 파일을 불러와, 요청이 들어오면 바로 예측 결과를 돌려주는 API를 만듭니다.

---
## 마무리 정리

| 확인할 내용 | 설명 |
|---|---|
| 이 파트의 핵심 | 분석과 모델을 "누구에게 무엇을 할지"라는 실행 목록으로 바꿨습니다. |
| 데이터가 문제를 정한다 | 재구매 3% → 고객 대신 상품(추천)과 판매자(이탈)로 문제를 다시 정의했습니다. |
| 모델 점검 | 배포용 모델 4개 합계 수백 KB. BERT는 배포하지 않습니다. |
| 최신 데이터 적용 | 학습 시점이 아니라 데이터의 마지막 시점으로 특징을 만들어 예측합니다. |
| 우선순위 | 이탈 확률 × 판매액처럼 **확률과 영향을 함께** 봅니다. |
| 다음 단계 | STEP2에서 모델을 API로 서빙합니다. |

In [26]:
# python
# # 저장 (학습 노트북에서)
# joblib.dump(묶음, "part7_1_sentiment_ml.pkl")

# 불러오기 (나중에, 또는 웹앱에서)
ml = joblib.load(f"{model_path}/part7_1_sentiment_ml.pkl")
ml["model"].predict(ml["tfidf"].transform(["produto muito bom"]))



array([1])